1) загрузка бд компас и таблиц

In [1]:
import sqlite3
import pandas as pd

DB_PATH = r"D:\diploma\jupyter lab\stage0 - data\compas.db"
conn = sqlite3.connect(DB_PATH)

# список таблиц
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)["name"].tolist()
print("Таблицы:", tables)

# читаем все таблицы в словарь датафреймов
dfs = {t: pd.read_sql(f'SELECT * FROM "{t}"', conn) for t in tables}
conn.close()

for name, df in dfs.items():
    print(f"{name}: {df.shape[0]} строк × {df.shape[1]} столбцов")

people        = dfs["people"]
compas        = dfs["compas"]
charge        = dfs["charge"]
casearrest    = dfs["casearrest"]
jailhistory   = dfs["jailhistory"]
prisonhistory = dfs["prisonhistory"]

Таблицы: ['casearrest', 'prisonhistory', 'jailhistory', 'compas', 'summary', 'charge', 'people']
casearrest: 128183 строк × 8 столбцов
prisonhistory: 4945 строк × 9 столбцов
jailhistory: 22111 строк × 7 столбцов
compas: 37578 строк × 21 столбцов
summary: 0 строк × 26 столбцов
charge: 148086 строк × 15 столбцов
people: 11757 строк × 41 столбцов


2 фильтрация аномалий

In [2]:
CUTOFF = pd.Timestamp("2016-04-01")   # конец периода наблюдения (последние даты в данных — 2016-03-30)
to_dt = lambda s: pd.to_datetime(s, errors="coerce")

def apply_filters(df, name, rules):
    """rules: {описание: булева маска строк, которые нужно УДАЛИТЬ}"""
    bad = pd.Series(False, index=df.index)
    print(f"\n=== {name}: {len(df)} строк")
    for desc, mask in rules.items():
        mask = mask.fillna(False)
        print(f"  {desc}: {mask.sum()} (новых: {(mask & ~bad).sum()})")
        bad |= mask
    out = df[~bad].reset_index(drop=True)
    print(f"  -> осталось {len(out)} (удалено {bad.sum()})")
    return out

clean = {}

# --- jailhistory ---
j = dfs["jailhistory"]
clean["jailhistory"] = apply_filters(j, "jailhistory", {
    "in_custody > out_custody": to_dt(j.in_custody) > to_dt(j.out_custody),
    "in_custody раньше даты рождения": to_dt(j.in_custody) < to_dt(j.dob),
})

# --- prisonhistory ---
pr = dfs["prisonhistory"]
clean["prisonhistory"] = apply_filters(pr, "prisonhistory", {
    "in_custody > out_custody": to_dt(pr.in_custody) > to_dt(pr.out_custody),
    "in_custody раньше даты рождения": to_dt(pr.in_custody) < to_dt(pr.dob),
})

# --- people ---
p = dfs["people"]
clean["people"] = apply_filters(p, "people", {
    "c_jail_in > c_jail_out": to_dt(p.c_jail_in) > to_dt(p.c_jail_out),
    "r_jail_in > r_jail_out": to_dt(p.r_jail_in) > to_dt(p.r_jail_out),
    "c_offense_date > c_arrest_date": to_dt(p.c_offense_date) > to_dt(p.c_arrest_date),
    "c_arrest_date > compas_screening_date": to_dt(p.c_arrest_date) > to_dt(p.compas_screening_date),
    "r_offense_date < compas_screening_date": to_dt(p.r_offense_date) < to_dt(p.compas_screening_date),
    "decile_score = -1 (нет оценки)": p.decile_score < 1,
    "is_recid = -1 (неизвестно)": p.is_recid < 0,
})

# --- compas ---
cp = dfs["compas"]
clean["compas"] = apply_filters(cp, "compas", {
    "decile_score = -1 (нет оценки)": cp.decile_score < 1,
    "полные дубликаты": cp.drop(columns="id").duplicated(),
})

# --- charge ---
ch = dfs["charge"]
clean["charge"] = apply_filters(ch, "charge", {
    "offense_date > date_charge_filed": to_dt(ch.offense_date) > to_dt(ch.date_charge_filed),
    "offense_date в будущем": to_dt(ch.offense_date) > CUTOFF,
    "полные дубликаты": ch.drop(columns="id").duplicated(),
})
clean["charge"]["filing_date"] = pd.NaT   # везде заглушка 1800-01-01

# --- casearrest ---
ca = dfs["casearrest"]
clean["casearrest"] = apply_filters(ca, "casearrest", {
    "arrest_date в будущем (2017, 2020, 2049)": to_dt(ca.arrest_date) > CUTOFF,
})

# --- оставляем во всех таблицах только людей, оставшихся в people ---
ids = set(clean["people"]["id"])
for t in ["compas", "charge", "casearrest", "jailhistory", "prisonhistory"]:
    before = len(clean[t])
    clean[t] = clean[t][clean[t].person_id.isin(ids)].reset_index(drop=True)
    print(f"{t}: удалено по person_id не из people — {before - len(clean[t])}")

people, compas, charge, casearrest, jailhistory, prisonhistory = (
    clean[t] for t in ["people", "compas", "charge", "casearrest", "jailhistory", "prisonhistory"])


=== jailhistory: 22111 строк
  in_custody > out_custody: 821 (новых: 821)
  in_custody раньше даты рождения: 0 (новых: 0)
  -> осталось 21290 (удалено 821)

=== prisonhistory: 4945 строк
  in_custody > out_custody: 0 (новых: 0)
  in_custody раньше даты рождения: 0 (новых: 0)
  -> осталось 4945 (удалено 0)

=== people: 11757 строк
  c_jail_in > c_jail_out: 302 (новых: 302)
  r_jail_in > r_jail_out: 113 (новых: 103)
  c_offense_date > c_arrest_date: 0 (новых: 0)
  c_arrest_date > compas_screening_date: 0 (новых: 0)
  r_offense_date < compas_screening_date: 0 (новых: 0)
  decile_score = -1 (нет оценки): 15 (новых: 15)
  is_recid = -1 (неизвестно): 719 (новых: 715)
  -> осталось 10622 (удалено 1135)

=== compas: 37578 строк
  decile_score = -1 (нет оценки): 21 (новых: 21)
  полные дубликаты: 0 (новых: 0)
  -> осталось 37557 (удалено 21)

=== charge: 148086 строк
  offense_date > date_charge_filed: 0 (новых: 0)
  offense_date в будущем: 0 (новых: 0)
  полные дубликаты: 13 (новых: 13)
  -> 

3) объединение заключений jail и prison в одну таблицу custody (одна строка — одно заключение)

In [3]:
# --- объединяем jail и prison (SQL поверх очищенных таблиц) ---
# привязка обвинений здесь НЕ делается: она целиком в следующей ячейке
# (этап 1 — арест в день входа, этап 2 — обвинение в день входа, этап 3 — окно 7 дней)
mem = sqlite3.connect(":memory:")
jailhistory.to_sql("jailhistory", mem, index=False)
prisonhistory.to_sql("prisonhistory", mem, index=False)

custody = pd.read_sql("""
SELECT person_id, first, last, dob, in_custody, out_custody, 'prison' AS source
FROM prisonhistory
UNION ALL
SELECT person_id, first, last, dob, in_custody, out_custody, 'jail' AS source
FROM jailhistory
ORDER BY person_id, in_custody;
""", mem)
mem.close()

custody["custody_id"] = range(len(custody))                                  # номер записи о заключении
custody["in_date"] = pd.to_datetime(custody["in_custody"]).dt.normalize()   # только дата, без времени

print(f"Записей о заключении: {len(custody)}")
print(custody["source"].value_counts().to_string())

Записей о заключении: 25052
source
jail      20273
prison     4779


In [4]:
WEEK = pd.Timedelta("7D")
DAY  = pd.Timedelta("1D")

custody["in_dt"]   = pd.to_datetime(custody["in_custody"])
custody["out_dt"]  = pd.to_datetime(custody["out_custody"])
custody["in_date"] = custody["in_dt"].dt.normalize()
ch = charge.assign(charge_date=pd.to_datetime(charge["date_charge_filed"]).dt.normalize())
ca = casearrest.assign(arrest_date=pd.to_datetime(casearrest["arrest_date"]).dt.normalize())

# --- разметка арестов: первый арест по делу или повторный (ордер / нарушение условий / неявка) ---
ca["first_arrest_date"] = ca.groupby(["person_id", "case_number"])["arrest_date"].transform("min")
ca["is_rearrest"] = ca["arrest_date"] > ca["first_arrest_date"]
case_degrees = (ch.groupby(["person_id", "case_number"])["charge_degree"]
                  .agg(lambda s: ", ".join(sorted(s.dropna().unique()))).to_dict())

arrest_dates = ca[["person_id", "case_number", "arrest_date", "is_rearrest"]].drop_duplicates()
print(f"Арестов (дат) всего: {len(arrest_dates)}, из них повторных по старому делу: {arrest_dates['is_rearrest'].sum()}")

charge_by_person = dict(tuple(ch.groupby("person_id")))
arrest_by_person = dict(tuple(ca.groupby("person_id")))
cust_by_person   = dict(tuple(custody.groupby("person_id")))
NO_CH, NO_AR = ch.iloc[0:0], ca.iloc[0:0]

links  = []     # (custody_id, charge_id, link_stage)
used   = {}     # charge_id -> custody_id, к которому обвинение привязано (только одно)
result = {}     # custody_id -> поля итога

def link(cid_custody, rows, stage):
    for cid in rows["id"]:
        links.append((cid_custody, cid, stage))
        used[cid] = cid_custody

def taken_by(rows):
    """к каким заключениям уже привязаны эти обвинения"""
    return ", ".join(sorted({str(used[i]) for i in rows["id"] if i in used}))

def join_ids(*parts):
    return ", ".join(sorted(set(x for p in parts if p for x in p.split(", ")))) or None

for r in custody.sort_values(["person_id", "in_dt"]).itertuples():
    c = charge_by_person.get(r.person_id, NO_CH)
    a = arrest_by_person.get(r.person_id, NO_AR)
    status, stage, reason, case_ids = [], None, None, None
    oldcase_ids, re_cases, re_first, re_degrees = None, None, None, None

    # --- этап 1: арест в день входа ---
    arr = a[a["arrest_date"] == r.in_date]
    if arr.empty:
        status.append("1: нет ареста в день входа")
    else:
        first_cases = sorted(arr.loc[~arr["is_rearrest"], "case_number"].unique())
        rearr       = arr[arr["is_rearrest"]].drop_duplicates("case_number")

        if len(rearr):                           # повторный арест по старому делу — запоминаем
            re_cases   = ", ".join(sorted(rearr["case_number"]))
            re_first   = ", ".join(sorted(rearr["first_arrest_date"].dt.strftime("%Y-%m-%d").unique()))
            re_degrees = "; ".join(f"{cn}: {case_degrees.get((r.person_id, cn), '?')}"
                                   for cn in sorted(rearr["case_number"]))

        if first_cases:                          # 1а: первый арест по делу -> обвинения дела
            s1 = c[c["case_number"].isin(first_cases)]
            s1_free = s1[~s1["id"].isin(used)]
            if len(s1_free):
                link(r.custody_id, s1_free, 1)
                stage, reason = 1, "new_offense"
                case_ids = ", ".join(sorted(s1_free["case_number"].unique()))
            elif s1.empty:
                status.append(f"1: первый арест по делу {', '.join(first_cases)}, но в charge нет обвинений по делу")
            else:
                oldcase_ids = join_ids(oldcase_ids, taken_by(s1))
                status.append(f"1: обвинения по делу {', '.join(first_cases)} уже привязаны "
                              f"к custody_id={taken_by(s1)} -> возможный олдкейс")
        else:                                    # 1б: только повторный арест -> обвинения НЕ привязываем
            status.append(f"1: в день входа только повторный арест по старому делу {re_cases} "
                          f"(первый арест {re_first}) -> обвинения дела не привязываются")

    # --- этап 2: обвинение в день входа (только ещё не привязанные) ---
    if stage is None:
        s2 = c[c["charge_date"] == r.in_date]
        s2_free = s2[~s2["id"].isin(used)]
        if len(s2_free):
            link(r.custody_id, s2_free, 2)
            stage, reason = 2, "new_offense"
            case_ids = ", ".join(sorted(s2_free["case_number"].unique()))
        elif s2.empty:
            status.append("2: нет обвинений в день входа")
        else:
            oldcase_ids = join_ids(oldcase_ids, taken_by(s2))
            status.append(f"2: обвинения в день входа уже привязаны к custody_id={taken_by(s2)}")

    # --- этап 3: ближайшее неучтённое обвинение в окне 7 дней до входа ---
    if stage is None:
        win = c[(c["charge_date"] >= r.in_date - WEEK) & (c["charge_date"] < r.in_date)]
        if win.empty:
            status.append("3: нет обвинений в окне 7 дней")
        else:
            free = win[~win["id"].isin(used)]
            if free.empty:
                oldcase_ids = join_ids(oldcase_ids, taken_by(win))
                status.append(f"3: все {len(win)} обвинений окна уже привязаны к custody_id={taken_by(win)} "
                              f"-> возможный олдкейс")
            else:
                nearest = free[free["charge_date"] == free["charge_date"].max()]   # только ближайший ко входу день
                link(r.custody_id, nearest, 3)
                stage, reason = 3, "new_offense"
                case_ids = ", ".join(sorted(nearest["case_number"].unique()))

    # --- итог 1б: нового обвинения нет, но был повторный арест -> старое дело без обвинений ---
    if stage is None and re_cases:
        stage, reason, case_ids = 1, "rearrest_old_case", re_cases

    # флаг олдкейса — только если новое обвинение так и не найдено
    oldcase = reason != "new_offense" and oldcase_ids is not None

    result[r.custody_id] = dict(link_stage=stage, reason_type=reason, case_ids=case_ids,
                                link_status="; ".join(status) or None,
                                possible_oldcase=oldcase, oldcase_custody_ids=oldcase_ids if oldcase else None,
                                rearrest_case_ids=re_cases, rearrest_first_arrest_date=re_first,
                                original_case_degrees=re_degrees)

res = pd.DataFrame.from_dict(result, orient="index").rename_axis("custody_id").reset_index()
custody = (custody.drop(columns=[c for c in res.columns if c != "custody_id" and c in custody.columns])
                  .merge(res, on="custody_id", how="left"))

# --- проверка перевода jail -> prison (для непривязанных prison) ---
def new_events(person_id, start_date, end_date):
    """число обвинений и арестов строго после start_date и до end_date включительно"""
    c = charge_by_person.get(person_id, NO_CH)
    a = arrest_by_person.get(person_id, NO_AR)
    return (((c["charge_date"] > start_date) & (c["charge_date"] <= end_date)).sum(),
            ((a["arrest_date"] > start_date) & (a["arrest_date"] <= end_date)).sum())

notes = {}
for r in custody[(custody["source"] == "prison") & custody["link_stage"].isna()].itertuples():
    pc = cust_by_person[r.person_id]
    jails = pc[(pc["source"] == "jail") & (pc["in_dt"] < r.in_dt)]
    if jails.empty:
        continue
    j = jails.loc[(r.in_dt - jails["out_dt"]).abs().idxmin()]      # jail с выходом ближе всего ко входу в prison
    gap = r.in_dt - j.out_dt
    n_ch, n_ar = new_events(r.person_id, j.in_dt.normalize(), r.in_date)
    if abs(gap) < DAY and n_ch == 0 and n_ar == 0:
        notes[r.custody_id] = (f"перевод jail -> prison: из custody_id={j.custody_id} "
                               f"(jail {j.in_dt:%Y-%m-%d} – {j.out_dt:%Y-%m-%d}, разрыв {gap})")
    elif abs(gap) < WEEK:
        why = []
        if abs(gap) >= DAY:
            why.append(f"разрыв выход/вход {gap}")
        if n_ch or n_ar:
            why.append(f"новые обвинения: {n_ch}, аресты: {n_ar}")
        notes[r.custody_id] = f"не перевод jail -> prison (custody_id={j.custody_id}): " + "; ".join(why)
custody["transfer_note"] = custody["custody_id"].map(notes)

# --- итоговая таблица: заключение + привязанные обвинения ---
links_df = pd.DataFrame(links, columns=["custody_id", "charge_id", "charge_link_stage"])
assert not links_df["charge_id"].duplicated().any(), "обвинение привязано к нескольким заключениям"
custody_charges = (
    custody
    .merge(links_df, on="custody_id", how="left")
    .merge(ch[["id", "charge_date", "case_number", "charge_number", "charge_degree", "charge", "statute"]]
             .rename(columns={"id": "charge_id"}), on="charge_id", how="left")
)
custody_charges["days_before_custody"] = (custody_charges["in_date"] - custody_charges["charge_date"]).dt.days

# --- статистика ---
n  = len(custody)
st = custody["link_stage"]
rt = custody["reason_type"]
print(f"\nВсего записей о заключении: {n}")
for label, mask in [("этап 1а — первый арест в день входа -> дело",      (st == 1) & (rt == "new_offense")),
                    ("этап 2 — обвинение в день входа",                  st == 2),
                    ("этап 3 — ближайшее в окне 7 дней",                 st == 3),
                    ("этап 1б — повторный арест, обвинения НЕ привязаны", rt == "rearrest_old_case"),
                    ("не привязано",                                    st.isna())]:
    print(f"  {label:<52} {mask.sum():>6} ({mask.mean():.2%})")

re_any = custody["rearrest_case_ids"].notna()
print(f"\nЗаключений, начавшихся в день повторного ареста: {re_any.sum()}")
print(f"  причина — новое обвинение:          {(re_any & (rt == 'new_offense')).sum()}")
print(f"  причина — старое дело без обвинений: {(re_any & (rt == 'rearrest_old_case')).sum()}")
print(f"\nВозможный олдкейс (новое обвинение не найдено): {custody['possible_oldcase'].sum()}")
print(f"Переводов jail -> prison найдено:               {custody['transfer_note'].str.startswith('перевод', na=False).sum()}")
print(f"Привязано обвинений: {len(links_df)} из {len(ch)} ({len(links_df) / len(ch):.2%})")

print("\nНе привязано по источнику:")
print(custody.groupby("source")["link_stage"].apply(lambda s: s.isna().mean()).map("{:.2%}".format))

C:\Users\Юлия\AppData\Local\Temp\ipykernel_16024\2454139259.py:1: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  WEEK = pd.Timedelta("7D")
C:\Users\Юлия\AppData\Local\Temp\ipykernel_16024\2454139259.py:2: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  DAY  = pd.Timedelta("1D")


Арестов (дат) всего: 69082, из них повторных по старому делу: 24264

Всего записей о заключении: 25052
  этап 1а — первый арест в день входа -> дело           13594 (54.26%)
  этап 2 — обвинение в день входа                          79 (0.32%)
  этап 3 — ближайшее в окне 7 дней                       1611 (6.43%)
  этап 1б — повторный арест, обвинения НЕ привязаны      3811 (15.21%)
  не привязано                                           5957 (23.78%)

Заключений, начавшихся в день повторного ареста: 5557
  причина — новое обвинение:          1746
  причина — старое дело без обвинений: 3811

Возможный олдкейс (новое обвинение не найдено): 45
Переводов jail -> prison найдено:               641
Привязано обвинений: 37366 из 141651 (26.38%)

Не привязано по источнику:
source
jail       5.91%
prison    99.56%
Name: link_stage, dtype: object


In [5]:
YEAR = pd.Timedelta("365D")

for col in ["stage4_arrest_date", "stage4_days_from_arrest"]:
    if col not in custody.columns:
        custody[col] = pd.NA

# переводы jail -> prison сюда не идут — они наследуют дело jail на этапе 5
is_transfer = custody["transfer_note"].str.startswith("перевод", na=False)
stage4 = custody[(custody["source"] == "prison") & custody["link_stage"].isna() & ~is_transfer] \
             .sort_values(["person_id", "in_dt"])

upd = {}        # custody_id -> поля, которые нужно обновить
for r in stage4.itertuples():
    c = charge_by_person.get(r.person_id, NO_CH)
    a = arrest_by_person.get(r.person_id, NO_AR)
    prev_status = r.link_status

    arr = a[(a["arrest_date"] >= r.in_date - YEAR) & (a["arrest_date"] <= r.in_date)]
    if arr.empty:
        upd[r.custody_id] = dict(link_status=f"{prev_status}; 4: нет арестов за год до входа в prison")
        continue

    linked, busy_cases, empty_cases = False, [], []
    # аресты от ближайшего ко входу к самому раннему
    for arrest_date, grp in arr.sort_values("arrest_date", ascending=False).groupby("arrest_date", sort=False):
        cases = sorted(grp["case_number"].unique())
        s4 = c[c["case_number"].isin(cases)]
        s4_free = s4[~s4["id"].isin(used)]
        if len(s4_free):
            link(r.custody_id, s4_free, 4)
            upd[r.custody_id] = dict(
                link_stage=4,
                case_ids=", ".join(sorted(s4_free["case_number"].unique())),
                stage4_arrest_date=arrest_date,
                stage4_days_from_arrest=(r.in_date - arrest_date).days,
            )
            linked = True
            break
        (busy_cases if len(s4) else empty_cases).extend(cases)

    if not linked:
        why = []
        if busy_cases:
            why.append(f"обвинения по делам {', '.join(sorted(set(busy_cases)))} уже привязаны "
                       f"к custody_id={taken_by(c[c['case_number'].isin(busy_cases)])}")
        if empty_cases:
            why.append(f"по делам {', '.join(sorted(set(empty_cases)))} нет обвинений в charge")
        upd[r.custody_id] = dict(link_status=f"{prev_status}; 4: аресты за год есть, но " + "; ".join(why))

# --- применяем обновления к custody ---
u = pd.DataFrame.from_dict(upd, orient="index")
for col in u.columns:
    vals = custody["custody_id"].map(u[col])
    m = vals.notna()
    custody.loc[m, col] = vals[m]

# --- пересобираем итоговую таблицу ---
links_df = pd.DataFrame(links, columns=["custody_id", "charge_id", "charge_link_stage"])
assert not links_df["charge_id"].duplicated().any(), "обвинение привязано к нескольким заключениям"
custody_charges = (
    custody
    .merge(links_df, on="custody_id", how="left")
    .merge(ch[["id", "charge_date", "case_number", "charge_number", "charge_degree", "charge", "statute"]]
             .rename(columns={"id": "charge_id"}), on="charge_id", how="left")
)
custody_charges["days_before_custody"] = (custody_charges["in_date"] - custody_charges["charge_date"]).dt.days

# --- статистика ---
st = custody["link_stage"]
print(f"Этап 4: проверено prison-строк {len(stage4)} (переводы пропущены: {(is_transfer & st.isna()).sum()}), "
      f"привязано {(st == 4).sum()} ({(st == 4).sum() / max(len(stage4), 1):.2%})")
print("Дней от ареста до входа в prison (этап 4):")
print(custody.loc[st == 4, "stage4_days_from_arrest"].astype(float).describe().round(1).to_string())
print("\nНе привязано по источнику:")
print(custody.groupby("source")["link_stage"].apply(lambda s: s.isna().mean()).map("{:.2%}".format))

C:\Users\Юлия\AppData\Local\Temp\ipykernel_16024\3438745063.py:1: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  YEAR = pd.Timedelta("365D")


Этап 4: проверено prison-строк 4117 (переводы пропущены: 641), привязано 2513 (61.04%)
Дней от ареста до входа в prison (этап 4):
count    2513.0
mean      104.9
std        91.7
min         1.0
25%        31.0
50%        77.0
75%       156.0
max       365.0

Не привязано по источнику:
source
jail       5.91%
prison    46.98%
Name: link_stage, dtype: object


In [6]:
PLACEHOLDER = pd.Timestamp("2020-01-01")   # заглушка «ещё сидит» в out_custody
custody = custody.reset_index(drop=True)

# ============ 1. удалить jail, начавшиеся внутри срока prison того же человека ============
pris = (custody.loc[(custody["source"] == "prison") & (custody["out_dt"] != PLACEHOLDER),
                    ["person_id", "custody_id", "in_dt", "out_dt"]]
               .rename(columns={"custody_id": "custody_id_prison",
                                "in_dt": "in_dt_prison",
                                "out_dt": "out_dt_prison"}))
jl = custody.loc[custody["source"] == "jail", ["person_id", "custody_id", "in_dt", "link_stage"]]
pairs = jl.merge(pris, on="person_id")
inside = pairs[(pairs["in_dt"] >= pairs["in_dt_prison"]) & (pairs["in_dt"] < pairs["out_dt_prison"])]

drop_ids = inside.loc[inside["link_stage"].isna(), "custody_id"].unique()
print(f"jail внутри срока prison: всего {inside['custody_id'].nunique()}, "
      f"из них непривязанных (удаляем) {len(drop_ids)}, "
      f"привязанных (оставлены) {inside.loc[inside['link_stage'].notna(), 'custody_id'].nunique()}")
custody = custody[~custody["custody_id"].isin(drop_ids)].reset_index(drop=True)

# ============ 2. этап 5: перевод jail -> prison наследует дело исходного jail ============
if "inherited_from_custody_id" not in custody.columns:
    custody["inherited_from_custody_id"] = pd.NA

tr = custody["link_stage"].isna() & custody["transfer_note"].str.startswith("перевод", na=False)
src_id = custody.loc[tr, "transfer_note"].str.extract(r"custody_id=(\d+)")[0].astype(int)

jail_cases = custody.set_index("custody_id")["case_ids"]
src_cases  = src_id.map(jail_cases)
ok  = src_cases.notna()
idx_ok, idx_bad = src_id.index[ok], src_id.index[~ok]

custody.loc[tr, "inherited_from_custody_id"] = src_id
custody.loc[idx_ok, "link_stage"] = 5
custody.loc[idx_ok, "case_ids"]   = src_cases[ok]
custody.loc[idx_bad, "link_status"] = (custody.loc[idx_bad, "link_status"].fillna("") +
    "; 5: перевод, но исходный jail custody_id=" + src_id[~ok].astype(str) + " сам не привязан к делу")

print(f"\nЭтап 5 (перевод jail -> prison): переводов {tr.sum()}, унаследовали дело {len(idx_ok)}, "
      f"исходный jail без дела {len(idx_bad)}")

# ============ пересобираем итоговую таблицу ============
links_df = pd.DataFrame(links, columns=["custody_id", "charge_id", "charge_link_stage"]).assign(inherited=False)
links_df = links_df[links_df["custody_id"].isin(custody["custody_id"])]
assert not links_df["charge_id"].duplicated().any(), "обвинение привязано к нескольким заключениям"

# обвинения переводов — ссылка на обвинения исходного jail (не новая привязка, обвинение остаётся за jail)
inherited_links = (
    custody.loc[custody["link_stage"] == 5, ["custody_id", "inherited_from_custody_id"]]
    .astype({"inherited_from_custody_id": int})
    .merge(links_df[["custody_id", "charge_id"]].rename(columns={"custody_id": "inherited_from_custody_id"}),
           on="inherited_from_custody_id")
    [["custody_id", "charge_id"]]
    .assign(charge_link_stage=5, inherited=True)
)
all_links = pd.concat([links_df, inherited_links], ignore_index=True)

custody_charges = (
    custody
    .merge(all_links, on="custody_id", how="left")
    .merge(ch[["id", "charge_date", "case_number", "charge_number", "charge_degree", "charge", "statute"]]
             .rename(columns={"id": "charge_id"}), on="charge_id", how="left")
)
custody_charges["days_before_custody"] = (custody_charges["in_date"] - custody_charges["charge_date"]).dt.days

# ============ статистика ============
st = custody["link_stage"]
print(f"\nВсего записей о заключении: {len(custody)}")
print(st.value_counts(dropna=False).sort_index().to_string())
print("\nНе привязано по источнику:")
print(custody.groupby("source")["link_stage"].apply(lambda s: s.isna().mean()).map("{:.2%}".format))

# ============ prom: непривязанные строки -> csv ============
prom = custody[custody["link_stage"].isna()].reset_index(drop=True)
print(f"\nНепривязанных строк: {len(prom)} из {len(custody)} ({len(prom) / len(custody):.2%})")
print(prom["source"].value_counts())
prom.to_csv("prom.csv", index=False, encoding="utf-8-sig")
print("Сохранено: prom.csv")

jail внутри срока prison: всего 36, из них непривязанных (удаляем) 12, привязанных (оставлены) 24

Этап 5 (перевод jail -> prison): переводов 641, унаследовали дело 603, исходный jail без дела 38

Всего записей о заключении: 25040
link_stage
1.0    17405
2.0       79
3.0     1611
4.0     2513
5.0      603
NaN     2829

Не привязано по источнику:
source
jail       5.86%
prison    34.36%
Name: link_stage, dtype: object

Непривязанных строк: 2829 из 25040 (11.30%)
source
prison    1642
jail      1187
Name: count, dtype: int64
Сохранено: prom.csv


In [7]:
# ================= этап 6: явные процедурные обвинения важнее старого дела =================
PROC_BEFORE = pd.Timedelta("30D")   # окно до входа
PROC_AFTER  = pd.Timedelta("1D")    # после входа — только задержка оформления (не нарушения в заключении)

def procedural_kind(df):
    """вид процедурного обвинения или NaN"""
    txt  = df["charge"].fillna("")
    stat = df["statute"].fillna("").astype(str)
    kind = pd.Series(pd.NA, index=df.index, dtype="object")
    rules = [
        ("нарушение условий освобождения до суда", txt.str.contains(r"(?i)viol\w*\s+pretrial\s+release")),
        ("нарушение охранного ордера",            stat.str.startswith(("741.31", "784.047"))
                                                   | txt.str.contains(r"(?i)viol\w*\s+(prot\w*\s+)?injunc")),
        ("неуважение к суду / неявка",            txt.str.contains(r"(?i)contempt|fail\w*\s+to\s+appear|bail\s+fail\s+appear|capias")),
        ("экстрадиция / розыск",                  stat.str.startswith("941.") | txt.str.contains(r"(?i)extradition|fugitive")
                                                   | (df["filing_type"].fillna("") == "Fugitive")),
        ("побег",                                 stat.str.startswith("944.40") | txt.str.contains(r"(?i)\bescape\b")),
        ("нарушение пробации (электронный браслет)", stat.str.startswith("948")),
    ]
    for name, mask in rules:
        kind = kind.where(~(mask & kind.isna()), name)
    return kind

ch["procedural_kind"] = procedural_kind(ch)
proc = ch[ch["procedural_kind"].notna()]
print("Процедурных обвинений в charge:")
print(proc["procedural_kind"].value_counts().to_string())

# --- кандидаты: всё, где причиной могло стать старое дело, и непривязанные ---
is_target = (custody["link_stage"].isna()
             | (custody["reason_type"] == "rearrest_old_case")
             | (custody["link_stage"] == 4))
targets   = custody[is_target]
proc_free = proc[~proc["id"].isin(used)]

pairs = targets[["custody_id", "person_id", "in_date", "link_stage", "reason_type"]].merge(
    proc_free[["id", "person_id", "charge_date", "procedural_kind", "case_number"]], on="person_id")
pairs["days_before_custody"] = (pairs["in_date"] - pairs["charge_date"]).dt.days
pairs = pairs[(pairs["charge_date"] >= pairs["in_date"] - PROC_BEFORE) &
              (pairs["charge_date"] <= pairs["in_date"] + PROC_AFTER)]

# ближайшие пары первыми; при равенстве — обвинение до входа, затем более раннее заключение
pairs = (pairs.assign(dist=pairs["days_before_custody"].abs(), after=pairs["days_before_custody"] < 0)
              .sort_values(["dist", "after", "in_date"]))

assigned_date, new_proc, taken = {}, [], set()
for p in pairs.itertuples():
    if p.id in taken:
        continue
    d = assigned_date.get(p.custody_id)
    if d is None or d == p.charge_date:   # одному заключению — обвинения только ближайшего дня
        assigned_date[p.custody_id] = p.charge_date
        new_proc.append(p)
        taken.add(p.id)

new_proc = pd.DataFrame(new_proc)
replaced_ids = set(new_proc["custody_id"]) if len(new_proc) else set()

for col in ["procedural_kind", "proc_days_before_custody", "replaced_reason", "replaced_case_ids", "replaced_degrees"]:
    if col not in custody.columns:
        custody[col] = pd.NA

# --- что было привязано к этим строкам раньше ---
old_links = pd.DataFrame([t for t in links if t[0] in replaced_ids],
                         columns=["custody_id", "charge_id", "charge_link_stage"])
old_info = (old_links.merge(ch[["id", "charge_degree"]].rename(columns={"id": "charge_id"}), on="charge_id")
                     .groupby("custody_id")
                     .agg(replaced_degrees=("charge_degree", lambda s: ", ".join(sorted(s.dropna().unique())))))

# --- снимаем старые привязки (этап 4) и освобождаем их обвинения ---
for _, cid, _ in [t for t in links if t[0] in replaced_ids]:
    used.pop(cid, None)
links = [t for t in links if t[0] not in replaced_ids]

# --- ставим процедурные ---
for p in new_proc.itertuples():
    links.append((p.custody_id, p.id, 6))
    used[p.id] = p.custody_id

if len(new_proc):
    before = custody.set_index("custody_id")
    g = new_proc.groupby("custody_id")
    upd = pd.DataFrame({
        "case_ids":                 g["case_number"].agg(lambda s: ", ".join(sorted(s.unique()))),
        "procedural_kind":          g["procedural_kind"].agg(lambda s: ", ".join(sorted(s.unique()))),
        "proc_days_before_custody": g["days_before_custody"].first(),
    }).join(old_info)
    upd["replaced_reason"]   = before.loc[upd.index, "link_stage"].map({4: "этап 4 (prison, арест за год)"}) \
                                   .fillna(before.loc[upd.index, "reason_type"]).fillna("не привязано")
    upd["replaced_case_ids"] = before.loc[upd.index, "case_ids"]

    m = custody["custody_id"].isin(upd.index)
    for col in upd.columns:
        custody.loc[m, col] = custody.loc[m, "custody_id"].map(upd[col])
    custody.loc[m, "link_stage"] = 6
    custody.loc[m, "link_status"] = (custody.loc[m, "link_status"].fillna("") +
        "; 6: привязано процедурное обвинение (" + custody.loc[m, "procedural_kind"] + ")").str.lstrip("; ")

# процедурный вид и для строк этапов 1–3, если среди их обвинений уже есть процедурное
kind_by_custody = (pd.DataFrame(links, columns=["custody_id", "charge_id", "s"])
                   .merge(ch[["id", "procedural_kind"]].rename(columns={"id": "charge_id"}), on="charge_id")
                   .dropna(subset=["procedural_kind"])
                   .groupby("custody_id")["procedural_kind"].agg(lambda s: ", ".join(sorted(s.unique()))))
m2 = custody["procedural_kind"].isna() & custody["custody_id"].isin(kind_by_custody.index)
custody.loc[m2, "procedural_kind"] = custody.loc[m2, "custody_id"].map(kind_by_custody)

# переводы (этап 5) наследуют актуальное дело исходного jail
m5 = custody["link_stage"] == 5
custody.loc[m5, "case_ids"] = custody.loc[m5, "inherited_from_custody_id"].astype(int).map(
    custody.set_index("custody_id")["case_ids"])

# ================= пересобираем итоговую таблицу =================
links_df = pd.DataFrame(links, columns=["custody_id", "charge_id", "charge_link_stage"]).assign(inherited=False)
links_df = links_df[links_df["custody_id"].isin(custody["custody_id"])]
assert not links_df["charge_id"].duplicated().any(), "обвинение привязано к нескольким заключениям"

inherited_links = (
    custody.loc[m5, ["custody_id", "inherited_from_custody_id"]]
    .astype({"inherited_from_custody_id": int})
    .merge(links_df[["custody_id", "charge_id"]].rename(columns={"custody_id": "inherited_from_custody_id"}),
           on="inherited_from_custody_id")[["custody_id", "charge_id"]]
    .assign(charge_link_stage=5, inherited=True)
)
all_links = pd.concat([links_df, inherited_links], ignore_index=True)

custody_charges = (
    custody
    .merge(all_links, on="custody_id", how="left")
    .merge(ch[["id", "charge_date", "case_number", "charge_number", "charge_degree", "charge", "statute", "procedural_kind"]]
             .rename(columns={"id": "charge_id", "procedural_kind": "charge_procedural_kind"}),
           on="charge_id", how="left")
)
custody_charges["days_before_custody"] = (custody_charges["in_date"] - custody_charges["charge_date"]).dt.days

# ================= статистика =================
st6 = custody["link_stage"] == 6
print(f"\nЭтап 6: процедурное обвинение привязано к {st6.sum()} заключениям")
print("  что было раньше:")
print(custody.loc[st6, "replaced_reason"].value_counts().to_string())
print("\n  вид процедурного обвинения:")
print(custody.loc[st6, "procedural_kind"].value_counts().to_string())
print("\n  дней от обвинения до входа (минус — после входа):")
print(custody.loc[st6, "proc_days_before_custody"].astype(float).describe().round(1).to_string())
print(f"\nСнято обвинений старых дел (этап 4): {len(old_links)}")

st = custody["link_stage"]
print(f"\nВсего записей о заключении: {len(custody)}")
print(st.value_counts(dropna=False).sort_index().to_string())

prom = custody[custody["link_stage"].isna()].reset_index(drop=True)
print(f"\nНепривязанных строк: {len(prom)} из {len(custody)} ({len(prom) / len(custody):.2%})")
prom.to_csv("prom.csv", index=False, encoding="utf-8-sig")
print("Сохранено: prom.csv")

C:\Users\Юлия\AppData\Local\Temp\ipykernel_16024\1492856021.py:2: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  PROC_BEFORE = pd.Timedelta("30D")   # окно до входа
C:\Users\Юлия\AppData\Local\Temp\ipykernel_16024\1492856021.py:3: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  PROC_AFTER  = pd.Timedelta("1D")    # после входа — только задержка оформления (не нарушения в заключении)
C:\Users\Юлия\AppData\Local\Temp\ipykernel_16024\1492856021.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  | txt.str.contains(r"(?i)viol\w*\s+(prot\w*\s+)?injunc")),


Процедурных обвинений в charge:
procedural_kind
нарушение охранного ордера                  475
нарушение условий освобождения до суда      169
побег                                        98
экстрадиция / розыск                         66
неуважение к суду / неявка                   26
нарушение пробации (электронный браслет)      3

Этап 6: процедурное обвинение привязано к 5 заключениям
  что было раньше:
replaced_reason
не привязано                     2
rearrest_old_case                2
этап 4 (prison, арест за год)    1

  вид процедурного обвинения:
procedural_kind
нарушение охранного ордера                4
нарушение условий освобождения до суда    1

  дней от обвинения до входа (минус — после входа):
count     5.0
mean     20.8
std       4.9
min      15.0
25%      16.0
50%      24.0
75%      24.0
max      25.0

Снято обвинений старых дел (этап 4): 1

Всего записей о заключении: 25040
link_stage
1.0    17403
2.0       79
3.0     1611
4.0     2512
5.0      603
6.0        5
NaN

In [8]:
# ================= этап 7: jail — арест в соседние дни (±3 дня от входа) =================
# оформление после полуночи / арест записан днём раньше или позже.
# Правила как на этапе 1: первый арест по делу -> свободные обвинения дела;
# только повторный арест по старому делу -> обвинения НЕ привязываются (rearrest_old_case).
NEAR = pd.Timedelta("3D")

for col in ["stage7_arrest_date", "stage7_days_from_arrest"]:
    if col not in custody.columns:
        custody[col] = pd.NA

stage7 = custody[(custody["source"] == "jail") & custody["link_stage"].isna()].sort_values(["person_id", "in_dt"])

upd = {}
for r in stage7.itertuples():
    c = charge_by_person.get(r.person_id, NO_CH)
    a = arrest_by_person.get(r.person_id, NO_AR)
    arr = a[(a["arrest_date"] >= r.in_date - NEAR) & (a["arrest_date"] <= r.in_date + NEAR)
            & (a["arrest_date"] != r.in_date)]
    if arr.empty:
        upd[r.custody_id] = dict(link_status=f"{r.link_status}; 7: нет арестов в ±3 дня от входа")
        continue

    # дни ареста от ближайшего ко входу; при равенстве — арест до входа
    arr = arr.assign(dist=(arr["arrest_date"] - r.in_date).dt.days.abs(),
                     after=arr["arrest_date"] > r.in_date)
    done, why, rearrest_hit = False, [], None
    for (dist, after, d), grp in arr.groupby(["dist", "after", "arrest_date"], sort=True):
        first_cases = sorted(grp.loc[~grp["is_rearrest"], "case_number"].unique())
        rearr = grp[grp["is_rearrest"]].drop_duplicates("case_number")
        if first_cases:
            s7 = c[c["case_number"].isin(first_cases)]
            s7_free = s7[~s7["id"].isin(used)]
            if len(s7_free):
                link(r.custody_id, s7_free, 7)
                upd[r.custody_id] = dict(link_stage=7, reason_type="new_offense",
                                         case_ids=", ".join(sorted(s7_free["case_number"].unique())),
                                         stage7_arrest_date=d, stage7_days_from_arrest=(r.in_date - d).days)
                done = True
                break
            why.append(f"арест {d:%Y-%m-%d} по делу {', '.join(first_cases)}: "
                       + (f"обвинения уже привязаны к custody_id={taken_by(s7)}" if len(s7) else "нет обвинений в charge"))
        if len(rearr) and rearrest_hit is None:
            rearrest_hit = (d, rearr)

    if done:
        continue
    if rearrest_hit is not None:            # нашёлся только повторный арест по старому делу
        d, rearr = rearrest_hit
        cases = sorted(rearr["case_number"])
        upd[r.custody_id] = dict(
            link_stage=7, reason_type="rearrest_old_case", case_ids=", ".join(cases),
            rearrest_case_ids=", ".join(cases),
            rearrest_first_arrest_date=", ".join(sorted(rearr["first_arrest_date"].dt.strftime("%Y-%m-%d").unique())),
            original_case_degrees="; ".join(f"{cn}: {case_degrees.get((r.person_id, cn), '?')}" for cn in cases),
            stage7_arrest_date=d, stage7_days_from_arrest=(r.in_date - d).days,
            link_status=f"{r.link_status}; 7: в ±3 дня только повторный арест по старому делу "
                        f"{', '.join(cases)} -> обвинения дела не привязываются")
    else:
        upd[r.custody_id] = dict(link_status=f"{r.link_status}; 7: " + "; ".join(why))

# --- применяем обновления к custody ---
u = pd.DataFrame.from_dict(upd, orient="index")
for col in u.columns:
    if col not in custody.columns:
        custody[col] = pd.NA
    vals = custody["custody_id"].map(u[col])
    m = vals.notna()
    custody.loc[m, col] = vals[m]

# --- переводы, у которых исходный jail теперь привязан, наследуют его дело (этап 5) ---
tr = custody["link_stage"].isna() & custody["transfer_note"].str.startswith("перевод", na=False)
src_id = custody.loc[tr, "transfer_note"].str.extract(r"custody_id=(\d+)")[0].astype(int)
src_cases = src_id.map(custody.set_index("custody_id")["case_ids"])
idx_ok = src_id.index[src_cases.notna()]
custody.loc[idx_ok, "inherited_from_custody_id"] = src_id[idx_ok]
custody.loc[idx_ok, "case_ids"]   = src_cases[idx_ok]
custody.loc[idx_ok, "link_stage"] = 5

# ================= пересобираем итоговую таблицу =================
m5 = custody["link_stage"] == 5
links_df = pd.DataFrame(links, columns=["custody_id", "charge_id", "charge_link_stage"]).assign(inherited=False)
links_df = links_df[links_df["custody_id"].isin(custody["custody_id"])]
assert not links_df["charge_id"].duplicated().any(), "обвинение привязано к нескольким заключениям"

inherited_links = (
    custody.loc[m5, ["custody_id", "inherited_from_custody_id"]]
    .astype({"inherited_from_custody_id": int})
    .merge(links_df[["custody_id", "charge_id"]].rename(columns={"custody_id": "inherited_from_custody_id"}),
           on="inherited_from_custody_id")[["custody_id", "charge_id"]]
    .assign(charge_link_stage=5, inherited=True)
)
all_links = pd.concat([links_df, inherited_links], ignore_index=True)

custody_charges = (
    custody
    .merge(all_links, on="custody_id", how="left")
    .merge(ch[["id", "charge_date", "case_number", "charge_number", "charge_degree", "charge", "statute", "procedural_kind"]]
             .rename(columns={"id": "charge_id", "procedural_kind": "charge_procedural_kind"}),
           on="charge_id", how="left")
)
custody_charges["days_before_custody"] = (custody_charges["in_date"] - custody_charges["charge_date"]).dt.days

# ================= статистика =================
st, rt = custody["link_stage"], custody["reason_type"]
s7 = st == 7
print(f"Этап 7: проверено непривязанных jail {len(stage7)}, привязано {s7.sum()} ({s7.sum() / max(len(stage7), 1):.2%})")
print(f"  новое обвинение (первый арест по делу): {(s7 & (rt == 'new_offense')).sum()}")
print(f"  повторный арест, обвинения НЕ привязаны: {(s7 & (rt == 'rearrest_old_case')).sum()}")
print("  дней от ареста до входа (минус — арест после входа):")
print(custody.loc[s7, "stage7_days_from_arrest"].astype(float).value_counts().sort_index().to_string())
print(f"\nПереводов, унаследовавших дело после этапа 7: {len(idx_ok)}")

print(f"\nВсего записей о заключении: {len(custody)}")
print(st.value_counts(dropna=False).sort_index().to_string())
print("\nНе привязано по источнику:")
print(custody.groupby("source")["link_stage"].apply(lambda s: s.isna().mean()).map("{:.2%}".format))

prom = custody[custody["link_stage"].isna()].reset_index(drop=True)
print(f"\nНепривязанных строк: {len(prom)} из {len(custody)} ({len(prom) / len(custody):.2%})")
prom.to_csv("prom.csv", index=False, encoding="utf-8-sig")
print("Сохранено: prom.csv")

C:\Users\Юлия\AppData\Local\Temp\ipykernel_16024\1619056201.py:5: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  NEAR = pd.Timedelta("3D")


Этап 7: проверено непривязанных jail 1185, привязано 289 (24.39%)
  новое обвинение (первый арест по делу): 96
  повторный арест, обвинения НЕ привязаны: 193
  дней от ареста до входа (минус — арест после входа):
stage7_days_from_arrest
-3.0      2
-2.0      1
-1.0     79
 1.0    200
 2.0      2
 3.0      5

Переводов, унаследовавших дело после этапа 7: 9

Всего записей о заключении: 25040
link_stage
1.0    17403
2.0       79
3.0     1611
4.0     2512
5.0      612
6.0        5
7.0      289
NaN     2529

Не привязано по источнику:
source
jail       4.42%
prison    34.17%
Name: link_stage, dtype: object

Непривязанных строк: 2529 из 25040 (10.10%)
Сохранено: prom.csv


In [9]:
# --- итоговый тип причины задержания для всех строк ---
rearrest_keys = set(zip(ca.loc[ca["is_rearrest"], "person_id"], ca.loc[ca["is_rearrest"], "arrest_date"]))

st = custody["link_stage"]
s4_arrest = pd.to_datetime(custody["stage4_arrest_date"]).dt.normalize()
s4_rearrest = [(p, d) in rearrest_keys for p, d in zip(custody["person_id"], s4_arrest)]

custody.loc[st == 4, "reason_type"] = "prison_by_arrest"
custody.loc[(st == 4) & pd.Series(s4_rearrest, index=custody.index), "reason_type"] = "revocation_old_case"
custody.loc[st == 5, "reason_type"] = "transfer"
custody.loc[st == 6, "reason_type"] = "procedural"
custody.loc[st.isna(), "reason_type"] = pd.NA

print(pd.crosstab(custody["reason_type"].fillna("не привязано"), custody["source"], margins=True))

custody.to_csv("custody_linked.csv", index=False, encoding="utf-8-sig")
custody_charges.to_csv("custody_charges.csv", index=False, encoding="utf-8-sig")
print("Сохранено: custody_linked.csv, custody_charges.csv")

source                jail  prison    All
reason_type                              
new_offense          15360      20  15380
prison_by_arrest         0     961    961
procedural               4       1      5
rearrest_old_case     4001       1   4002
revocation_old_case      0    1551   1551
transfer                 0     612    612
не привязано           896    1633   2529
All                  20261    4779  25040
Сохранено: custody_linked.csv, custody_charges.csv


In [ ]:
# ================= типы преступлений по тексту обвинения (15 типов) =================
# Правила — из crime_categories.py (те же, что в прошлых версиях пайплайна).
# categorize      -> 9 типов: NoCharge, Sex, Traffic, Violent, Drug, Weapon, Fraud, Property, Other
# categorize_fine -> 15 типов: 9 базовых + ResistPolice, PublicOrder, Tampering, Justice, Conspiracy, Minors
# Правила проверяются по порядку, срабатывает первое совпадение (регистр не важен).
import re

RULES = [
    ("NoCharge", r"no charge"),
    ("Sex", r"lewd|lasciv|sex|prostitu|indecent expos|voyeur|porn|obscen|molest|travel.*minor|child exploit|computer for child"),
    ("Traffic", r"\bdui\b|d\.u\.i|\bdwi\b|\bdwls\b|driv|licen|\bdl\b|\blic\b|motorcycle|vehicle insurance|motor veh ins"
                r"|insur.*card|proof of invalid insur|flee|elud|leav.*(scen|acc)|lve/scen|reckless driv|register vehicle"
                r"|tag|sticker|boater|load|tri-rail|taxi"
                # нарушения ПДД (часто идут сопутствующими обвинениями в том же деле)
                r"|safety belt|stbelt|seat ?belt|child restraint|pers/inj/prot|submit insurance"
                r"|stop sign|signal|red light|flashing|yield|ftyrow|\bspeed|too fast|following too closely"
                r"|\blanes?\b|\bturn|backing|stop/stand|stop stand|stand/park|imp park|improp park|park handicap|park/obstruct|hov|sunscr|wndow|windshield|window"
                r"|due care|unsafe/improp|unsafe/improper|improp equip|lights|lamps|mcycle|bike|ped fail|x-walk"
                r"|refuse submit blood|breath test|render aid|rept accd|road surface|sounds>1000|display regist"),
    ("Violent", r"murder|manslaughter|homicide|home invasion|kidnap|carjack|robbery|battery|\bbatt\b|batt/|assault|false imprison|stalk"
                r"|child abuse|cruelty|neglect|abuse|shoot|throw.*missile|throw in occup|harm public servant"
                r"|injunc|dom vi|domestic|dating viol|threat|interference with custody|culpable negligence"
                r"|exposes culpable"),
    ("Drug", r"cocaine|cannabis|heroin|meth|oxycod|hydrocod|hydromorph|alprazolam|clonazepam|diazepam|lorazepam"
             r"|amphet|morphine|codeine|fentanyl|methadone|bupren|carisoprodol|phentermine|oxymorph|mdma|lsd"
             r"|ecstasy|ethylone|butylone|xlr11|pyrrolid|piperaz|pipe\b|controlled? subst|contr sub|control subst"
             r"|cont subs|paraphernalia|tetrahydro|\brx\b|prescript|drug|traff|jwh|fluoro|gamma-hydroxy|butanediol"
             r"|steroid|trenbolon|amobarbital|methox|contrft schd|countrft|cntrft|canbs|fubinaca|synthetic"),
    ("Weapon", r"firearm|weapon|\bwep\b|f/arm|concealed|ammun|destruct device|bomb"),
    ("Fraud", r"forg|utter|fraud|credit card|\bcc\b|personal id|identity|false name|false own|counterfeit|worthless"
              r"|defraud|launder|structuring|scanning device|innkeeper|ID card|unlaw.*\bid\b|unlaw use false"
              r"|imperson|police badge|simulation|draft|trade secret|intellectual|bribery|telemarket"
              r"|white collar|bad check|false info|false 911|false crime|giving false|posses.*altered bills|unlaw pos of prson|hire|lease"),
    ("Property", r"theft|burgl|stolen|mischief|shoplift|arson|trespass|tresspass|retail|cart|tools|fire extin|\bburg\b|burg/"
                 r"|altered id|alt vin|railroad|fuel|deprive"),
]
COMPILED = [(name, re.compile(rx, re.I)) for name, rx in RULES]
OTHER = "Other"
UNKNOWN = "Unknown"


def categorize(desc):
    if pd.isna(desc) or not str(desc).strip():
        return UNKNOWN
    for name, rx in COMPILED:
        if rx.search(str(desc)):
            return name
    return OTHER


# ------------------------------------------------------------------------------
# Дробление Other (решение Greer, 08.10.2026). Применяется только к обвинениям,
# которые правила выше отнесли к Other. Сначала исправления явных ошибок
# группировки (обвинение относится к уже существующему типу), затем новые
# состояния. Порядок важен: срабатывает первое совпадение.
FIX_RULES = [
    ("Traffic", r"interlock device|writ acc rpt"),
    ("PublicOrder", r"harassing phone|phone ring repeat"),
    ("Property", r"tresps"),
    ("Justice", r"escape|contraband|intro cont|extradition|contempt|fail.*appear|handcuff key|elec monitor"
                r"|dna sample|fail register|failure to register|change address/name|report change/residence"
                r"|accessory after|perjury|false affidavit|false report|false reports|false id given to leo"
                r"|post bond|viol court order"),
    ("Tampering", r"tamper|witness|victim|harass|fabricat|retaliat"),
    ("Minors", r"minor|delinq|juvenile|compulsory|school function|child<6|crlty twrd child"),
    ("Conspiracy", r"conspir|crim cnsp|solicitation on|attempt/solicitation|criminal attempt|principal in the first|2 way device"),
    ("Drug", r"sale/del/pos|temazepam|pethidine|benocy|psilocyn|s/m/d/p|cnsp traf|halluc|cann|contrl sub|contr sub"
             r"|marijuana|chlordiazepox|pb-22|nbome|ur-144|propoxyphene|paraph|flurazepam|testost|w/int near a park"
             r"|inhale harmful"),
    ("Fraud", r"corrupt public servant|pawn|secondhand|reg metals|exploit elderly|cred|false receipt|counterf|false insurance"
              r"|false statemnt ins|false personation|false documents|rico|patient brokering|money trans|currenc|bank bills"
              r"|copy of sound|audio visual|sound articles|vin plate|altered motor vehicle|id\" removed|alter id markings"
              r"|work comp ins|computer user|personating officer"),
    ("Property", r"coin operated|copper|phone cables|phone/power|petit frm merchant|property not structure|fence"
                 r"|dairy case|altering \(outdoor advert\)"),
    ("Weapon", r"f/a serial|f/a altered|short barl|self propelled knife"),
    ("Violent", r"affray|engage in fighting|fight/brawl|agg bodily inj|att murd|baiting animals|hiv|dog bite"),
    ("Traffic", r"imp start|engine run|proof of insur|airless tires|improper pass|improper stop|improp|unauthorizd person drv"
                r"|headset|improper parking|ped|walk left|move over|block intersection|disobey green|double parked|muffler"
                r"|stop lamp|moped|special haz|fast for cond|horn|atv|lghts|rear view|passing|no pass|bicycle|school bus"
                r"|no d/l|rear light|rear lamp|windscreen|exhaust|defective equip|bumper|veh reg|safety +belt|accident report"
                r"|accid|habit traf|head lamp|headlamp|^infraction$|unsafe equip|too loud from vehicle|cmv|rr barrier|rr crossing"
                r"|lim acces|obstrct veh|crash|motor vehicle noise|operate motor vehicle|veh on public beach|100' of intersection"
                r"|financial responsibility|racing|upper beam|vehicle dealer|aircraft registration"),
    ("ResistPolice", r"resist|obstruct|obey police|disobey officer|refuse stop when ordered|depriv leo|k9|police"),
    ("PublicOrder", r"prowl|loiter|disorderly|doc/|bev|alcohol|alch|container|cont in m/v|cntnr|contnr|drink|drunk|intox"
                    r"|911|false alarm|sleep|lodg|litter|liter|urinate|panhandl|solic|noise|gambl|bookmak|lottery|ordinance"
                    r"|unlawful assembly|disturbing the peace|disturb education|city park|park hours|beach|indecen|wear mask"
                    r"|vaccine|plumbing|construct contract|mobile unit|guests|bigamy|optical dev|desertion|electrical|county"),
]
FIX_COMPILED = [(name, re.compile(rx, re.I)) for name, rx in FIX_RULES]
NEW_STATES = ["ResistPolice", "PublicOrder", "Tampering", "Justice", "Conspiracy", "Minors"]


def categorize_fine(desc):
    base = categorize(desc)
    if base != OTHER:
        return base
    for name, rx in FIX_COMPILED:
        if rx.search(str(desc)):
            return name
    return OTHER

# --- применяем к обвинениям ---
cat15 = {d: categorize_fine(d) for d in ch["charge"].dropna().unique()}
ch["charge_category"] = ch["charge"].map(cat15).fillna(UNKNOWN)

custody_charges["charge_category"] = custody_charges["charge"].map(cat15)
custody_charges.loc[custody_charges["charge_id"].notna() & custody_charges["charge_category"].isna(),
                    "charge_category"] = UNKNOWN

# --- по заключению: набор типов его обвинений ---
cats_by_custody = (custody_charges.dropna(subset=["charge_category"])
                   .groupby("custody_id")["charge_category"]
                   .agg(lambda s: ", ".join(sorted(s.unique()))))
custody["charge_categories"] = custody["custody_id"].map(cats_by_custody)
custody["n_charge_categories"] = custody["charge_categories"].str.count(",").add(1).fillna(0).astype(int)

# --- статистика ---
linked = custody_charges[custody_charges["charge_id"].notna()]
print(f"Формулировок обвинений: {len(cat15)}, из них осталось в Other: "
      f"{sum(v == OTHER for v in cat15.values())}")
print("\nПривязанные обвинения по типам (15):")
print(linked["charge_category"].value_counts().to_string())
print("\nЗаключения по типам (заключение считается в каждом своём типе):")
print(custody["charge_categories"].dropna().str.split(", ").explode().value_counts().to_string())
print(f"\nЗаключений без обвинений (нет типа): {custody['charge_categories'].isna().sum()} "
      f"(в т.ч. rearrest_old_case: {(custody['charge_categories'].isna() & (custody['reason_type'] == 'rearrest_old_case')).sum()})")

custody.to_csv("custody_linked.csv", index=False, encoding="utf-8-sig")
custody_charges.to_csv("custody_charges.csv", index=False, encoding="utf-8-sig")
print("Сохранено: custody_linked.csv, custody_charges.csv")

In [10]:
# ================= фильтрация: prison с входом до 2013 года =================
# до 2013 года в базе почти нет арестов и обвинений, привязывать такие сроки не к чему.
# Убранные строки сохраняются отдельно — они понадобятся позже для ковариат (история заключений).
DATA_START = pd.Timestamp("2013-01-01")

pre2013 = (custody["source"] == "prison") & (custody["in_dt"] < DATA_START)
prison_before_2013 = custody[pre2013].reset_index(drop=True)
prison_before_2013_charges = custody_charges[custody_charges["custody_id"].isin(prison_before_2013["custody_id"])] \
                                 .reset_index(drop=True)

print(f"prison с входом до {DATA_START:%Y-%m-%d}: {len(prison_before_2013)} из {(custody['source'] == 'prison').sum()} prison")
print(f"  из них привязано: {prison_before_2013['link_stage'].notna().sum()}, "
      f"не привязано: {prison_before_2013['link_stage'].isna().sum()}")
print(f"  срок ещё шёл на {DATA_START:%Y-%m-%d} (выход позже): {(prison_before_2013['out_dt'] >= DATA_START).sum()}")
print(f"  людей: {prison_before_2013['person_id'].nunique()}")

prison_before_2013.to_csv("prison_before_2013.csv", index=False, encoding="utf-8-sig")
prison_before_2013_charges.to_csv("prison_before_2013_charges.csv", index=False, encoding="utf-8-sig")
print("Сохранено: prison_before_2013.csv, prison_before_2013_charges.csv")

# --- убираем из рабочих таблиц ---
custody         = custody[~pre2013].reset_index(drop=True)
custody_charges = custody_charges[custody_charges["custody_id"].isin(custody["custody_id"])].reset_index(drop=True)
prom            = custody[custody["link_stage"].isna()].reset_index(drop=True)

# --- статистика после фильтрации ---
print(f"\nВсего записей о заключении: {len(custody)}")
print(pd.crosstab(custody["reason_type"].fillna("не привязано"), custody["source"], margins=True))
print(f"\nНепривязанных строк: {len(prom)} из {len(custody)} ({len(prom) / len(custody):.2%})")
print(prom["source"].value_counts().to_string())

custody.to_csv("custody_linked.csv", index=False, encoding="utf-8-sig")
custody_charges.to_csv("custody_charges.csv", index=False, encoding="utf-8-sig")
prom.to_csv("prom.csv", index=False, encoding="utf-8-sig")
print("Сохранено: custody_linked.csv, custody_charges.csv, prom.csv")

prison с входом до 2013-01-01: 3663 из 4779 prison
  из них привязано: 2329, не привязано: 1334
  срок ещё шёл на 2013-01-01 (выход позже): 141
  людей: 1359
Сохранено: prison_before_2013.csv, prison_before_2013_charges.csv

Всего записей о заключении: 21377
source                jail  prison    All
reason_type                              
new_offense          15360       2  15362
prison_by_arrest         0      94     94
procedural               4       0      4
rearrest_old_case     4001       0   4001
revocation_old_case      0     109    109
transfer                 0     612    612
не привязано           896     299   1195
All                  20261    1116  21377

Непривязанных строк: 1195 из 21377 (5.59%)
source
jail      896
prison    299
Сохранено: custody_linked.csv, custody_charges.csv, prom.csv
